# Mini-Batch Gradient Descent

Mini-batch gradient descent is a successor to [Batch Gradient Descent (BGD)](batch_gradient_descent.ipynb). BGD runs the train loop (i.e., Forward → Backpropagation → Update parameters weight and biases) on the **entire dataset**. This can be computationally intensive especially with a larger dataset. Mini-batch gradient descent **divides** the training data into **small mini-batches**, which means you don't have to load the entire dataset into the **RAM** to train the model, only the current mini-batch that the model is training on. *Mini-batch gradient descent converges faster than BGD, and is more computationally efficient!*

Mini-batch gradient descent is widely used.


$$\theta := \theta - \frac{\alpha}{m} \sum_{i=1}^m \Delta_{\theta} C(\theta;x^{(i)}, y^{(i)}) $$

Note:

---

- 💡 Mini-batch gradient descent is a **strategy**, **not an optimizer**. It is the process of feeding a model small batches of the dataset. The **Optimizer** (like [Adam](adam.ipynb) or [SGD](stochastic_gradient_descent.ipynb))  is the algorithm that updates the parameters using a batch's information.
  - Mini-batches' job is to calculate the average gradient: $\frac{1}{m} \sum_{i=1}^m \Delta_{\theta} C(\theta;x^{(i)}, y^{(i)})$ (without $\alpha$)
  - The optimizer's job is what performs the parameter update ($\theta := \theta - \alpha * ...$) using that gradient. This is the (SGD) update rule.

---

- $\theta$ is called 'theta', it is a parameter (weight or bias). When we calculate the gradient for the weights this formula looks like $w:= w- \frac{\alpha}{m} \sum_{i=1}^m \Delta_{w} C(w;x^{(i)}, y^{(i)})$, this shows that you are taking the gradient with respect to $w$ (using $\Delta w$) to update $w$. For the bias it looks like: $b:= b- \frac{\alpha}{m} \sum_{i=1}^m \Delta_{b} C(b;x^{(i)}, y^{(i)})$
- $\alpha$ is the learning rate, which is typically a small value like 0.1, 0.01, or smaller.
- $\Delta$ is called 'delta'
- $C$ represents the Cost function.
- $m$ represents mini-batch size. 
- $\sum_{i=1}^m$ is a summation that calculates the average gradient only for the $m$ samples in the current mini-batch.
- $\Delta_{\theta} C$ is the gradient. It is a vector that points in the direction of the steepest ascent, i.e., the direction that increases the cost function the most.
  - Since our goal is to **minimize** the cost, we need to move in the opposite direction of $\Delta_{\theta} C$ to go 'downhill' on the cost landscape, and reduce the cost. This part is done here $\theta - ...$, by subtracting the gradient from our current parameter $\theta$

- $x^{(i)}, y^{(i)}$ are the input features of the $i$-th sample in a mini-batch *(e.g., I have a dataset of 512 images of dogs, I divide that dataset into 8 mini-batches, so that each batch contains 64 images. One dog image in a mini of 64 images is the $x^{(i)}$, and its label is the $y^{(i)}$)*.

- $\Delta_{\theta} C(\theta;x^{(i)}, y^{(i)})$ is the gradient of the cost function with respect to the $\theta$ for the $i$-th sample.


### How To Use Mini-Batch Gradient Descent

**Steps:**
1. Split the dataset: We have $512$ images of dogs, divide into batches of $64$ (**Mini-batch size**).
   - How to pick a good Mini-batch size: Generally use a size between $32$ and $128$ (like $64$), this will result in more frequent updates (the process of forward → backpropagation → update parameters), but it can also introduce noise into the optimization process. This noise is a benefit as it prevents the model from getting stuck in local minima. If you are using powerful GPUs or TPUs, you can use a higher batch size (e.g., a size like $256$, $512$, or larger), this will result in more stable updates but can increase memory consumption.
2. The (forward → backpropagation → update parameters) process is computed **per mini-batch** instead of per the entire dataset.
3. Epochs: An epoch refers to one complete pass through the entire dataset, not just one mini-batch. At the start of each epoch the entire dataset is typically reshuffled to ensure that the model does not overfit to a specific order of data!

### Steps in code (from scratch):

In [1]:
import numpy as np

batch_size = 64
num_epochs = 5
learning_rate = 0.01

# Initialize weights and biases
weights = np.random.randn(32 * 32, 1)
biases = np.random.randn(1)

# 1. Load dataset
num_dog_images = 512
dummy_dog_images = (
    np.random.randint(  # These images are not dog images just dummy pixels values!
        low=0, high=256, size=(num_dog_images, 32, 32)  # Each image is 32x32 pixels
    )
)
dummy_dog_labels = np.random.randint(
    low=0, high=2, size=(num_dog_images)
)  # e.g., 0 or 1, if 1 the image contains a dog, otherwise 0


# 2. Function for splitting the dataset into Mini-batches
def get_minibatch(features, labels, batch_size):
    """
    Generator that yields Mini-batches from the dataset.

    Returns: A mini-batch contains tuples of an image and its corresponding label.
    """
    num_samples = features.shape[0]
    for i in range(0, num_samples, batch_size):
        yield (features[i : i + batch_size], labels[i : i + batch_size])


# 3. Train loop
for epoch in range(num_epochs):
    print(f"\n --- Epoch {epoch+1}/{num_epochs} ---")

    # Shuffle the dataset for every epoch
    indices = np.arange(num_dog_images)
    np.random.shuffle(indices)
    dummy_dog_images = dummy_dog_images[indices]
    dummy_dog_labels = dummy_dog_labels[indices]

    minibatch_generator = get_minibatch(dummy_dog_images, dummy_dog_labels, batch_size)

    # Iterate directly over the batches generated
    for batch_index, (images_batch, labels_batch) in enumerate(minibatch_generator):
        # One mini-batch contains 64 images

        # --- FORWARD PROPAGATION ---
        # Reshape images so it can be multiplied by the weights
        images_batch_reshaped = images_batch.reshape(images_batch.shape[0], -1)
        dummy_predictions = np.dot(images_batch_reshaped, weights) + biases

        # --- BACKPROPAGATION ---
        # Calculate the average gradient (Δθ) over the mini-batch (m)
        # (1/m) * ∑(ΔC)
        # These are dummy values to not over-complicate this notebook!
        dummy_avg_grad_w = np.random.randn(32*32, 1)
        dummy_avg_grad_b = np.random.randn(1)

        # --- UPDATE PARAMETERS (W & B) ---
        # θ := θ - alpha * (average_gradient)
        weights = weights - learning_rate * dummy_avg_grad_w
        biases = biases - learning_rate * dummy_avg_grad_b

        print(
            f"    Processed batch {batch_index+1}: Images shape: {images_batch.shape}, Labels shape: {labels_batch.shape}"
        )


 --- Epoch 1/5 ---
    Processed batch 1: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 2: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 3: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 4: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 5: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 6: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 7: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 8: Images shape: (64, 32, 32), Labels shape: (64,)

 --- Epoch 2/5 ---
    Processed batch 1: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 2: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 3: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 4: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 5: Images shape: (64, 32, 32), Labels shape: (64,)
    Processed batch 6: Images shape: 

### Steps in code (with PyTorch)

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64
num_epochs = 5
learning_rate = 1e-3

# 1. Load dataset
num_dog_images = 512
dummy_dog_images = torch.randint(
    # These images are not dog images just dummy pixels values!
    low=0,
    high=256,
    size=(num_dog_images, 32, 32),  # Each image is 32x32 pixels
    dtype=torch.float32,
)
dummy_dog_labels = torch.randint(
    low=0, high=2, size=(num_dog_images,1), dtype=torch.float32
)  # e.g., 0 or 1, if 1 the image contains a dog, otherwise 0


# 2. Splitting dataset into mini-batches: Create PyTorch TensorDataset
dog_dataset = TensorDataset(dummy_dog_images, dummy_dog_labels)
# Create the DataLoader
dog_dataloader = DataLoader(
    dataset=dog_dataset,
    batch_size=batch_size,
    shuffle=True,  # Shuffle per epoch
    drop_last=True,
)

# Define Torch model
model = nn.Sequential(nn.Flatten(), nn.Linear(in_features=32 * 32, out_features=1))
loss_func = nn.BCEWithLogitsLoss()

# Optimizer: The Mini-batch gradient descent
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# 3. Train loop
for epoch in range(num_epochs):
    print(f"\n --- Epoch {epoch+1}/{num_epochs} ---")

    # Iterate directly over the DataLoader
    for batch_index, (images_batch, labels_batch) in enumerate(dog_dataloader):
        # One mini-batch containing 64 images

        # --- FORWARD PROPAGATION ---
        dummy_predictions = model(images_batch)
        loss = loss_func(dummy_predictions, labels_batch)

        # --- BACKPROPAGATION ---
        optimizer.zero_grad()  # Clear old gradients

        # This calculates the gradients: (Δθ * C), and computes (1/m) * ∑(ΔC) automatically.
        loss.backward()

        # --- UPDATE PARAMETERS (W & B) ---
        optimizer.step()  # Computes: θ := θ - alpha * (average_gradient)

        print(
            f"    Processed batch {batch_index+1}: Images shape: {images_batch.shape}, Labels shape: {labels_batch.shape}"
        )


 --- Epoch 1/5 ---
    Processed batch 1: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 2: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 3: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 4: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 5: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 6: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 7: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 8: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])

 --- Epoch 2/5 ---
    Processed batch 1: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size([64, 1])
    Processed batch 2: Images shape: torch.Size([64, 32, 32]), Labels shape: torch.Size